# Trade the 8-K

Gator Quant Hacks 2026. Massive Challenge.

## Submission

This notebook uses the 8-K category share_repurchase_program.
The strategy is the covered call.
The expiry bucket is 3-6m.
The OTM distance is 5 percent.
ENTRY is post.

The notebook calculates all five strategies.
Do not change the covered call after the scoreboard.
The cash-secured put is the other strategy for the same category.

## Data

The notebook uses two data sets from api.massive.com:
- Filings and Disclosures for 8-K events
- Option contracts and daily option bars

Put MASSIVE_API_KEY in the .env file.
Do not put the key in a cell.
The API cache is the folder .massive_cache.


## Strategy list

The notebook can calculate these strategies:
- Long call: buy one at-the-money call
- Covered call: hold the stock and sell one call
- Protective put: hold the stock and buy one put
- Collar: hold the stock, buy one put, and sell one call
- Cash-secured put: sell one put

If a strategy needs 100 shares, the notebook uses a synthetic stock.
The synthetic stock comes from the at-the-money call and the at-the-money put.


## What you submit

1. This notebook. The notebook must run from a clean kernel with an API key.
2. A write-up of two pages or less.
3. The function run_study. The judge will call this function on a new date window.

The judge does not score profit and loss.


## Setup

Use Python 3.10 or a later version.
Use the kernel that has the packages from requirements.txt.

1. Put MASSIVE_API_KEY in the .env file in this folder.
2. Open this notebook.
3. Run all cells from the top.

The first run can take 10 minutes or more.
Later runs use the cache.


In [1]:
# Optional: only if you did NOT run setup.sh / setup.ps1. Installs into this kernel; restart the kernel afterwards.
# %pip install -q -r requirements.txt


## 1. API client

The function api_get reads one page from the Massive REST API.
The function writes each response to disk.
The function api_get_all reads all pages.


In [2]:
import os, json, time, hashlib, re
from dataclasses import dataclass, field
from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from pandas.tseries.holiday import (AbstractHolidayCalendar, Holiday, nearest_workday, USMartinLutherKingJr,
                                    USPresidentsDay, GoodFriday, USMemorialDay, USLaborDay, USThanksgivingDay)
from pandas.tseries.offsets import CustomBusinessDay

BASE_URL = "https://api.massive.com"


def load_api_key(name: str = "MASSIVE_API_KEY") -> str:
    """Environment first, then a `.env` file beside the notebook, then a prompt. Never paste a key into a cell."""
    key = (os.environ.get(name) or "").strip()
    env_file = Path(".env")
    if not key and env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.strip().startswith(f"{name}="):
                key = line.split("=", 1)[1].strip().strip('"').strip("'")
    if not key:
        key = getpass(f"{name} (https://massive.com/dashboard/keys): ").strip()
    return key


API_KEY = load_api_key()
SESSION = requests.Session()
SESSION.headers["Authorization"] = f"Bearer {API_KEY}"

CACHE_DIR = Path(".massive_cache")
CACHE_DIR.mkdir(exist_ok=True)


def api_get(path_or_url: str, params: dict | None = None) -> dict:
    """GET one page from the Massive REST API. Cached on disk by the full URL, so reruns are free."""
    url = path_or_url if path_or_url.startswith("http") else BASE_URL + path_or_url
    full_url = requests.Request("GET", url, params=params).prepare().url
    cache_file = CACHE_DIR / (hashlib.sha1(full_url.encode()).hexdigest() + ".json")
    if cache_file.exists():
        return json.loads(cache_file.read_text())
    for attempt in range(10):
        resp = SESSION.get(full_url, timeout=60)
        if resp.status_code in (429, 500, 502, 503, 504):   # rate-limited or a transient server error
            retry_after = resp.headers.get("Retry-After", "")
            time.sleep(float(retry_after) if retry_after.isdigit() else min(2 ** attempt, 20))
            continue
        break
    resp.raise_for_status()
    payload = resp.json()
    cache_file.write_text(json.dumps(payload))
    return payload


def api_get_all(path: str, params: dict | None = None, max_pages: int = 500) -> list[dict]:
    """Follow `next_url` pagination and return every result row."""
    payload = api_get(path, params)
    rows = list(payload.get("results") or [])
    pages = 1
    while payload.get("next_url") and pages < max_pages:
        payload = api_get(payload["next_url"])
        rows.extend(payload.get("results") or [])
        pages += 1
    return rows


print(f"API key loaded (ends {API_KEY[-4:]}); cache at {CACHE_DIR.resolve()}")


API key loaded (ends a8ul); cache at D:\dev\CodeOfDuty\Hackathons\gqh-massive-8k-starter-kit\.massive_cache


## 2. Configuration

Change only the values in the next cell.
Do not change the list of horizons.

EVENT_TAG is share_repurchase_program.
PRIMARY_STRATEGY is covered_call.
Do not change PRIMARY_STRATEGY after the scoreboard.

The in-sample window is STUDY_START to STUDY_END.
The out-of-sample window is OOS_START to OOS_END.
The sealed window is HOLDOUT_START to HOLDOUT_END.
Set RUN_HOLDOUT to True only for the sealed test.

TOP_100 is a fixed list of 100 companies.
ENTRY post is the close of the filing session.
ENTRY pre is the close of the session before the filing session.
The tables use ENTRY.

OTM_PCT is the distance of the sold call and of the put.
FETCH_ACCEPTANCE_TIMES moves t_0 when the 8-K is accepted at 16:00 ET or after.


In [3]:
# ---- The 8-K event. One tertiary_category tag from the taxonomy (printed in the next section). ------
EVENT_TAG = "share_repurchase_program"
PRIMARY_STRATEGY = "covered_call"   # Do not change this after the scoreboard.
WINNER = PRIMARY_STRATEGY
NEIGHBOR_STRATEGY = "cash_secured_put"  # Same category and same parameters.

# Windows. Do not change these dates. The judge changes only the sealed window.
STUDY_START, STUDY_END = "2024-01-01", "2025-12-31"      # in-sample (plan has ~2 years of options history)
OOS_START, OOS_END = "2026-01-01", "2026-08-31"          # out-of-sample, run in this notebook
HOLDOUT_START, HOLDOUT_END = "2023-06-01", "2023-08-31"  # sealed: judges change these and flip RUN_HOLDOUT

# Universe. Fixed list of 100 companies. See the appendix.
TOP_100 = """
AAPL ABBV ABT ACN ADBE AIG AMD AMGN AMT AMZN AVGO AXP BA BAC BK BKNG BLK BMY BRK.B C
CAT CHTR CL CMCSA COF COP COST CRM CSCO CVS CVX DE DHR DIS DUK EMR FDX GD GE GILD
GM GOOGL GS HD HON IBM INTC INTU ISRG JNJ JPM KO LIN LLY LMT LOW MA MCD MDLZ MDT
MET META MMM MO MRK MS MSFT NEE NFLX NKE NOW NVDA ORCL PEP PFE PG PLTR PM PYPL QCOM
RTX SBUX SCHW SO T TGT TMO TMUS TSLA TXN UBER UNH UNP UPS USB V VZ WFC WMT XOM
""".split()
assert len(TOP_100) == 100 and len(set(TOP_100)) == 100

# ---- Expiry buckets: name -> (min, max, target) calendar days to expiry from the pre-event session.
EXPIRY_BUCKETS = {
    "1m":   (21, 45, 30),
    "2m":   (46, 80, 60),
    "3-6m": (90, 180, 120),
}
BASELINE_BUCKET = "3-6m"     # the headline bucket: the out-of-sample test is on options 3-6 months out

# ---- Fixed horizons (sessions after the filing session). Fixed for every team. Do not change. -----
HORIZONS = [1, 2, 3, 5, 10, 21, 42, 63]

# ---- Strategy parameters -----------------------------------------------------------------------
OTM_PCT = 0.05              # covered call / collar sell the call ~5% above spot; protective put / collar / cash-secured put use the put ~5% below
OTM_GRID = [0.03, 0.05, 0.10]   # neighbours for the parameter-sensitivity check; OTM_PCT must be one of them
ENTRY = "post"              # "post": close of the filing session (the tradeable rule). "pre": close of the session before (hypothetical: was it priced?)
RISK_FREE = 0.04            # flat carry rate used in put-call parity to recover spot from the chain

# ---- Mechanics -----------------------------------------------------------------------------------
STRIKE_WINDOW = 0.25        # keep strikes within ±25% of spot once spot is known
MAX_STALE_SESSIONS = 3      # a leg's last trade may be at most this many sessions old to count as a mark
MAX_EVENTS = None           # e.g. 15 for a smoke test; None = all events

# ---- Stretch toggles -----------------------------------------------------------------------------
RUN_PLACEBO = True          # the same measurement on ordinary days for the same names
N_PLACEBO = 120
PLACEBO_GAP_DAYS = 30
FETCH_ACCEPTANCE_TIMES = True   # pull <ACCEPTANCE-DATETIME> from EDGAR to fix same-day vs next-day entry
SEC_USER_AGENT = (os.environ.get("SEC_USER_AGENT") or "").strip() or (
    "GatorQuantHacks gqh8koptions@gmail.com"
)  # EDGAR needs CompanyName and an email. A noreply address gets HTTP 403.
RUN_HOLDOUT = False         # judges flip this

assert OTM_PCT in OTM_GRID and BASELINE_BUCKET in EXPIRY_BUCKETS and ENTRY in ("pre", "post")

# ---- Catch impossible dates (e.g. "2024-06-31") here rather than deep in the pipeline -----------
for _name in ("STUDY_START", "STUDY_END", "OOS_START", "OOS_END", "HOLDOUT_START", "HOLDOUT_END"):
    try:
        pd.Timestamp(globals()[_name])
    except ValueError:
        raise ValueError(f"{_name} = {globals()[_name]!r} is not a real date (check the day of the month)") from None
assert STUDY_START < STUDY_END and OOS_START < OOS_END and HOLDOUT_START < HOLDOUT_END, "each window's start must be before its end"


### Disclosure taxonomy

The next cell reads all tertiary_category tags.
The cell shows the family of EVENT_TAG.


In [4]:
taxonomy = pd.DataFrame(api_get_all("/stocks/taxonomies/vX/disclosures", {"limit": 1000}))
print(f"{len(taxonomy)} tertiary tags, {taxonomy.secondary_category.nunique()} secondary, "
      f"{taxonomy.primary_category.nunique()} primary")
family = taxonomy.loc[taxonomy.tertiary_category == EVENT_TAG, "secondary_category"].iloc[0]
with pd.option_context("display.max_colwidth", 100):
    display(taxonomy[taxonomy.secondary_category == family]
            [["primary_category", "secondary_category", "tertiary_category", "description"]]
            .reset_index(drop=True))


119 tertiary tags, 29 secondary, 8 primary


,primary_category,secondary_category,tertiary_category,description
0,capital_and_financing,shareholder_returns,dividend_declaration,"Cash dividend, special dividend, or distribution declaration with amount, record date, and payme..."
1,capital_and_financing,shareholder_returns,dividend_policy_change,"Material change to dividend policy including suspension, reduction, or significant increase."
2,capital_and_financing,shareholder_returns,share_repurchase_program,"Share buyback program authorization, expansion, or update with amount and timing."
3,capital_and_financing,shareholder_returns,tender_offer,"Tender offer to acquire outstanding securities (equity, debt, or trust preferred) with pricing a..."
4,capital_and_financing,shareholder_returns,stock_split,"Forward or reverse stock split with ratio, dates, and share impact."


## 3. Trading calendar

Each horizon counts NYSE sessions.
The notebook builds the calendar from weekdays minus NYSE closures.
The notebook has no stock bar feed.


In [5]:
class NYSEHolidays(AbstractHolidayCalendar):
    """NYSE full-day closures. Differs from the federal calendar: Good Friday closed, Columbus and
    Veterans Day open, no observance when New Year's Day falls on a Saturday."""
    rules = [
        Holiday("New Year's Day", month=1, day=1,
                observance=lambda d: d + pd.Timedelta(days=1) if d.weekday() == 6 else d),
        USMartinLutherKingJr, USPresidentsDay, GoodFriday, USMemorialDay,
        Holiday("Juneteenth", month=6, day=19, start_date="2022-01-01", observance=nearest_workday),
        Holiday("Independence Day", month=7, day=4, observance=nearest_workday),
        USLaborDay, USThanksgivingDay,
        Holiday("Christmas Day", month=12, day=25, observance=nearest_workday),
        Holiday("National day of mourning, President Carter", year=2025, month=1, day=9),
    ]


def trading_sessions(start, end) -> pd.DatetimeIndex:
    holidays = NYSEHolidays().holidays(pd.Timestamp(start) - pd.Timedelta(days=7), pd.Timestamp(end) + pd.Timedelta(days=7))
    return pd.bdate_range(start, end, freq=CustomBusinessDay(holidays=holidays))


CAL = trading_sessions("2021-06-01", "2027-12-31")
TODAY = pd.Timestamp.today().normalize()
LAST_SESSION = CAL[CAL.searchsorted(TODAY, side="right") - 1]


def session_on_or_after(day) -> pd.Timestamp:
    return CAL[CAL.searchsorted(pd.Timestamp(day), side="left")]


def session_before(day) -> pd.Timestamp:
    return CAL[CAL.searchsorted(pd.Timestamp(day), side="left") - 1]


def sessions_between(a, b) -> int:
    """Number of sessions strictly after `a` up to and including `b`."""
    return int(CAL.searchsorted(pd.Timestamp(b), side="right") - CAL.searchsorted(pd.Timestamp(a), side="right"))


print(f"{len(CAL)} sessions in the calendar; last completed session {LAST_SESSION.date()}. "
      f"Sessions per year: " + ", ".join(f"{y}: {((CAL.year == y)).sum()}" for y in range(2022, 2027)))


1655 sessions in the calendar; last completed session 2026-10-02. Sessions per year: 2022: 251, 2023: 250, 2024: 252, 2025: 250, 2026: 251


## 4. Events

The notebook reads all filers for EVENT_TAG, then keeps TOP_100.
One event is one pair of cik and filing_date.

t_0 is the first NYSE session on the filing_date or after the filing_date.
If the 8-K is accepted at 16:00 ET or after, t_0 is the next session.
t_pre is the session before t_0.
The notebook reads the option chain on t_pre.


In [6]:
def normalize_ticker(t) -> str | None:
    """Filings write share classes as BRK/B or BRK.B; Massive's options use BRK.B as the underlying."""
    if not isinstance(t, str) or not t.strip():
        return None
    return t.strip().upper().replace("/", ".")


def fetch_disclosures(tag: str, start: str, end: str) -> pd.DataFrame:
    rows = api_get_all("/stocks/filings/8-K/vX/disclosures", {
        "tertiary_category": tag, "filing_date.gte": start, "filing_date.lte": end,
        "limit": 1000, "sort": "filing_date.asc",
    })
    df = pd.DataFrame(rows)
    if not df.empty:
        df["filing_date"] = pd.to_datetime(df["filing_date"])
    return df


def build_events(tag: str, start: str, end: str, universe: list[str]) -> pd.DataFrame:
    """One row per (filer, filing date) in the universe, with the filing session and the session before it."""
    raw = fetch_disclosures(tag, start, end)
    print(f"{len(raw):,} '{tag}' disclosures across all filers, {start}..{end}")
    ex = raw.explode("tickers").rename(columns={"tickers": "ticker"})
    ex["ticker"] = ex["ticker"].map(normalize_ticker)
    ex = ex[ex["ticker"].isin(universe)]
    ev = (ex.sort_values(["cik", "filing_date"])
            .groupby(["cik", "filing_date"], as_index=False)
            .agg(ticker=("ticker", "first"), accession_number=("accession_number", "first"),
                 filing_url=("filing_url", "first"), supporting_text=("supporting_text", "first"))
            .sort_values("filing_date").reset_index(drop=True))
    ev["t_0"] = ev["filing_date"].map(session_on_or_after)      # the filing session: first session on or after the filing date
    ev["t_pre"] = ev["t_0"].map(session_before)                 # the session before it: the chain here cannot know the news
    ev["days_since_prev_event"] = ev.groupby("ticker")["filing_date"].diff().dt.days
    return ev


def fetch_acceptance_time(filing_url: str) -> pd.Timestamp | None:
    """Acceptance timestamp (US/Eastern) from the EDGAR submission header; None if not found."""
    if not isinstance(filing_url, str) or not filing_url.strip():
        return None
    key = CACHE_DIR / ("sec_" + hashlib.sha1(filing_url.encode()).hexdigest() + ".txt")
    if key.exists():
        head = key.read_text(errors="replace")
    else:
        resp = requests.get(filing_url, headers={"User-Agent": SEC_USER_AGENT}, timeout=30)
        if resp.status_code != 200:
            raise RuntimeError(f"EDGAR HTTP {resp.status_code}")
        head = resp.content[:8192].decode("utf-8", errors="replace")
        key.write_text(head, errors="replace")
        time.sleep(0.12)
    m = re.search(r"<ACCEPTANCE-DATETIME>(\d{14})", head)
    return pd.Timestamp(m.group(1)) if m else None


def apply_acceptance_times(ev: pd.DataFrame) -> pd.DataFrame:
    """If EDGAR accepted the filing at or after 16:00 ET, the tradeable session is the next one."""
    ev = ev.copy()
    if ev.empty or not FETCH_ACCEPTANCE_TIMES or "filing_url" not in ev.columns:
        return ev
    accepted, n_fail = [], 0
    for u in ev["filing_url"]:
        try:
            accepted.append(fetch_acceptance_time(u))
        except Exception as exc:
            n_fail += 1
            if n_fail <= 3:
                print(f"  EDGAR fetch failed ({type(exc).__name__}: {exc})")
            accepted.append(None)
    ev["accepted_at"] = accepted
    hour = pd.to_datetime(ev["accepted_at"], errors="coerce").dt.hour
    after_close = hour.ge(16).fillna(False)
    if after_close.any():
        ev.loc[after_close, "t_0"] = [CAL[CAL.searchsorted(d, side="right")] for d in ev.loc[after_close, "t_0"]]
        ev["t_pre"] = ev["t_0"].map(session_before)
    n_known = int(pd.Series(accepted).notna().sum())
    print(f"Acceptance times: {n_known}/{len(ev)} parsed, {n_fail} fetch failures, "
          f"{int(after_close.sum())} after 16:00 ET (those t_0 moved to the next session).")
    return ev


events = build_events(EVENT_TAG, STUDY_START, STUDY_END, TOP_100)
events = apply_acceptance_times(events)
if MAX_EVENTS:
    events = events.head(MAX_EVENTS).copy()
print(f"{len(events)} in-sample events across {events.ticker.nunique()} of the top 100; "
      f"{(events.filing_date != events.t_0).sum()} have t_0 ≠ filing_date (weekend/holiday or post-16:00 ET acceptance)")
display(events.groupby(events.filing_date.dt.year).size().rename("events").to_frame().T)
with pd.option_context("display.max_colwidth", 110):
    display(events[["ticker", "filing_date", "t_pre", "t_0", "supporting_text"]].sample(5, random_state=1))


2,108 'share_repurchase_program' disclosures across all filers, 2024-01-01..2025-12-31


Acceptance times: 0/36 filings after 16:00 ET; those t_0 moved to the next session.
36 in-sample events across 26 of the top 100; 0 were filed on a non-session day (weekend or holiday)


filing_date,2024,2025
events,16,20


,ticker,filing_date,t_pre,t_0,supporting_text
30,CRM,2025-09-03,2025-09-02,2025-09-03,"On September 3, 2025, the Company announced an increase of $20.0 billion in the amount authorized for repu..."
34,EMR,2025-11-05,2025-11-04,2025-11-05,"On November 5, 2025, Emerson Electric Co. (the ""Company"") announced that its Board of Directors has author..."
28,BAC,2025-07-23,2025-07-22,2025-07-23,"the Corporation's Board of Directors (the ""Board"") authorized the Corporation's $40 billion common stock r..."
3,DIS,2024-02-07,2024-02-06,2024-02-07,"The Board of Directors of the Company approved a new program, effective February 7, 2024, for the repurcha..."
19,SCHW,2025-02-10,2025-02-07,2025-02-10,"On February 9, 2025, the Company entered into a stock repurchase agreement with the selling stockholder pu..."


## 5. Option chain and spot

For each event the notebook reads the chain on t_pre.

1. Find spot. Spot is K * e^(-rT) + C - P on the at-the-money pair.
2. Select the expiry in each bucket.
3. Select the strikes. K is the paired strike that is nearest to spot.

The function locate_spot repeats until the strike does not change.


In [7]:
def option_bars(opt_ticker: str, start, end) -> pd.DataFrame:
    """Daily bars for one contract: index = session, columns close and volume. Only sessions where it traded."""
    rows = api_get_all(f"/v2/aggs/ticker/{opt_ticker}/range/1/day/{pd.Timestamp(start):%Y-%m-%d}/{pd.Timestamp(end):%Y-%m-%d}",
                       {"adjusted": "false", "sort": "asc", "limit": 50000})
    if not rows:
        return pd.DataFrame(columns=["close", "volume"], index=pd.DatetimeIndex([], name="session"))
    idx = (pd.to_datetime([r["t"] for r in rows], unit="ms", utc=True)
             .tz_convert("America/New_York").normalize().tz_localize(None))
    return pd.DataFrame({"close": [float(r["c"]) for r in rows], "volume": [float(r.get("v") or 0) for r in rows]},
                        index=pd.DatetimeIndex(idx, name="session"))


def fetch_chain(ticker: str, as_of: pd.Timestamp, dte_lo: int, dte_hi: int) -> pd.DataFrame:
    """Every standard contract that existed on `as_of` with an expiry `dte_lo`..`dte_hi` days out."""
    rows = api_get_all("/v3/reference/options/contracts", {
        "underlying_ticker": ticker, "as_of": as_of.strftime("%Y-%m-%d"),
        "expiration_date.gte": (as_of + pd.Timedelta(days=dte_lo)).strftime("%Y-%m-%d"),
        "expiration_date.lte": (as_of + pd.Timedelta(days=dte_hi)).strftime("%Y-%m-%d"),
        "limit": 1000,
    })
    chain = pd.DataFrame(rows)
    if chain.empty:
        return chain
    if "shares_per_contract" in chain:
        chain = chain[chain["shares_per_contract"].fillna(100) == 100]      # drop post-split non-standard series
    chain = chain[["ticker", "contract_type", "strike_price", "expiration_date"]].copy()
    chain["expiration_date"] = pd.to_datetime(chain["expiration_date"])
    chain["dte"] = (chain["expiration_date"] - as_of).dt.days
    chain["strike_price"] = chain["strike_price"].astype(float)
    return chain.reset_index(drop=True)


def paired_strikes(e: pd.DataFrame) -> np.ndarray:
    both = e.groupby("strike_price")["contract_type"].nunique()
    return both[both == 2].index.to_numpy(dtype=float)


def contract(e: pd.DataFrame, strike: float, kind: str) -> str:
    return e[(e.strike_price == strike) & (e.contract_type == kind)]["ticker"].iloc[0]


def last_close_on_or_before(opt_ticker: str, day: pd.Timestamp, lookback_days: int = 7) -> float | None:
    bars = option_bars(opt_ticker, day - pd.Timedelta(days=lookback_days), day)
    return float(bars["close"].iloc[-1]) if len(bars) else None


def locate_spot(chain: pd.DataFrame, day: pd.Timestamp, max_iter: int = 8) -> dict | None:
    """Recover the stock price on `day` from the option chain alone, via put-call parity.

    At any strike K with both a call and a put, S = K·e^(−rT) + C − P. The strike that minimises |C − P|
    is at-the-money, and the estimate is best there because both legs are liquid. We start at the
    median strike of the nearest expiry (new weekly series are listed around the current price, so that
    median is already close), then step to the strike nearest our estimate until it stops moving.
    """
    near = chain[chain.dte >= 3]
    if near.empty:
        return None
    near = near[near.dte == near.dte.min()]
    strikes = paired_strikes(near)
    if len(strikes) < 3:
        return None
    T = near.dte.iloc[0] / 365
    k, tried, est = float(np.median(strikes)), set(), None
    k = strikes[np.abs(strikes - k).argmin()]
    for _ in range(max_iter):
        tried.add(k)
        c = last_close_on_or_before(contract(near, k, "call"), day)
        p = last_close_on_or_before(contract(near, k, "put"), day)
        if c is None or p is None:                                   # an illiquid strike: try the next-nearest untried one
            rest = [s for s in strikes if s not in tried]
            if not rest:
                break
            k = rest[int(np.abs(np.array(rest) - k).argmin())]
            continue
        est = k * np.exp(-RISK_FREE * T) + c - p
        k_new = strikes[np.abs(strikes - est).argmin()]
        if k_new == k or k_new in tried:
            break
        k = k_new
    if est is None:
        return None
    return {"spot": float(est), "strike": float(k), "expiry": near.expiration_date.iloc[0], "dte": int(near.dte.iloc[0])}


def pick_expiry(chain: pd.DataFrame, lo: int, hi: int, target: int) -> pd.Timestamp | None:
    cand = chain[(chain.dte >= lo) & (chain.dte <= hi)]
    if cand.empty:
        return None
    dte_of = cand.groupby("expiration_date")["dte"].first()
    ok = [x for x, g in cand.groupby("expiration_date") if len(paired_strikes(g)) >= 3]   # needs a real chain, not a stub
    if not ok:
        return None
    return min(ok, key=lambda x: abs(dte_of[x] - target))


def select_strikes(e: pd.DataFrame, spot: float, otm_pcts: list[float]) -> dict[str, float] | None:
    """ATM strike K (nearest to spot, both legs listed) plus, for each OTM level, the call strike U at or
    above spot·(1+pct) and the put strike L at or below spot·(1−pct)."""
    both = paired_strikes(e)
    both = both[(both >= spot * (1 - STRIKE_WINDOW)) & (both <= spot * (1 + STRIKE_WINDOW))]
    if len(both) == 0:
        return None
    calls = np.sort(e.loc[e.contract_type == "call", "strike_price"].unique())
    puts = np.sort(e.loc[e.contract_type == "put", "strike_price"].unique())
    out = {"K": float(both[np.abs(both - spot).argmin()])}
    for pct in otm_pcts:
        up, dn = calls[calls >= spot * (1 + pct)], puts[puts <= spot * (1 - pct)]
        out[f"U{pct}"] = float(up.min()) if len(up) else float(calls.max())
        out[f"L{pct}"] = float(dn.max()) if len(dn) else float(puts.min())
    return out


### Price each event

The function price_events does the four steps for each event and each bucket.
The function returns the priced legs or a drop reason.
The notebook does not fill missing marks.


In [8]:
@dataclass
class Leg:
    ticker: str
    kind: str                 # "call" or "put"
    strike: float
    bars: pd.DataFrame        # close, volume by session (only sessions with trades)

    def mark(self, day: pd.Timestamp) -> float:
        """Last traded close on or before `day`, if it is at most MAX_STALE_SESSIONS sessions old."""
        b = self.bars.loc[: pd.Timestamp(day)]
        if b.empty or sessions_between(b.index[-1], day) > MAX_STALE_SESSIONS:
            return np.nan
        return float(b["close"].iloc[-1])

    def volume_on(self, day: pd.Timestamp) -> float:
        return float(self.bars["volume"].get(pd.Timestamp(day), 0.0))


@dataclass
class PricedEvent:
    """One event × one expiry bucket: the chain-derived spot, the expiry and every leg a strategy may need."""
    ticker: str
    event_date: pd.Timestamp
    t_pre: pd.Timestamp
    t_0: pd.Timestamp
    bucket: str
    expiry: pd.Timestamp
    expiry_session: pd.Timestamp        # last session on or before expiry
    spot_pre: float                     # chain-implied spot on t_pre
    strikes: dict[str, float]
    legs: dict[str, Leg]                # "C_K", "P_K", "C_U0.05", "P_L0.05", ...

    def marks(self, day) -> dict[str, float]:
        return {name: leg.mark(day) for name, leg in self.legs.items()}

    def synthetic_spot(self, day, m: dict[str, float] | None = None) -> float:
        """Stock price implied by the ATM pair on `day`: K·e^(−rT) + C_K − P_K. Exact at expiry."""
        m = m if m is not None else self.marks(day)
        T = max((self.expiry - pd.Timestamp(day)).days, 0) / 365
        return self.strikes["K"] * np.exp(-RISK_FREE * T) + m["C_K"] - m["P_K"]


def price_event(ticker: str, t_pre: pd.Timestamp, t_0: pd.Timestamp, event_date: pd.Timestamp,
                buckets: dict, otm_pcts: list[float]) -> tuple[list[PricedEvent], list[str]]:
    dte_hi = max(b[1] for b in buckets.values())
    chain = fetch_chain(ticker, t_pre, 2, dte_hi)
    if chain.empty:
        return [], ["no option chain as of the pre-event session"]
    loc = locate_spot(chain, t_pre)
    if loc is None:
        return [], ["could not recover spot from the chain (no liquid near-dated pair)"]
    priced, notes = [], []
    for name, (lo, hi, target) in buckets.items():
        expiry = pick_expiry(chain, lo, hi, target)
        if expiry is None:
            notes.append(f"{name}: no expiry {lo}-{hi} days out")
            continue
        e = chain[chain.expiration_date == expiry]
        strikes = select_strikes(e, loc["spot"], otm_pcts)
        if strikes is None:
            notes.append(f"{name}: no paired strikes near spot")
            continue
        wanted = {"C_K": ("call", strikes["K"]), "P_K": ("put", strikes["K"])}
        for pct in otm_pcts:
            wanted[f"C_U{pct}"] = ("call", strikes[f"U{pct}"])
            wanted[f"P_L{pct}"] = ("put", strikes[f"L{pct}"])
        legs = {}
        for key, (kind, k) in wanted.items():
            tk = contract(e, k, kind)
            legs[key] = Leg(tk, kind, k, option_bars(tk, t_pre - pd.Timedelta(days=10), expiry))
        pe = PricedEvent(ticker, event_date, t_pre, t_0, name, expiry, CAL[CAL.searchsorted(expiry, side="right") - 1],
                         loc["spot"], strikes, legs)
        if np.isnan(pe.legs["C_K"].mark(t_pre)) or np.isnan(pe.legs["P_K"].mark(t_pre)):
            notes.append(f"{name}: ATM pair did not trade on or near the pre-event session")
            continue
        priced.append(pe)
    return priced, notes


def price_events(ev: pd.DataFrame, buckets: dict = None, otm_pcts: list[float] = None, label: str = "events") -> tuple[list[PricedEvent], pd.DataFrame]:
    buckets = buckets or EXPIRY_BUCKETS
    otm_pcts = otm_pcts or OTM_GRID
    priced, dropped, t0 = [], [], time.time()
    for n, row in enumerate(ev.itertuples(index=False), 1):
        got, notes = price_event(row.ticker, row.t_pre, row.t_0, row.event_date if hasattr(row, "event_date") else row.filing_date,
                                 buckets, otm_pcts)
        priced += got
        dropped += [(row.ticker, row.t_0, note) for note in notes]
        if n % 25 == 0:
            print(f"  {label}: {n}/{len(ev)} events, {time.time() - t0:.0f}s")
    print(f"{label}: {len(ev)} events -> {len(priced)} priced (event, bucket) pairs; {len(dropped)} drops; {time.time() - t0:.0f}s")
    return priced, pd.DataFrame(dropped, columns=["ticker", "t_0", "reason"])


def summarize_priced(priced: list[PricedEvent]) -> pd.DataFrame:
    rows = []
    for pe in priced:
        m = pe.marks(pe.t_pre)
        rows.append({"ticker": pe.ticker, "event_date": pe.event_date, "t_pre": pe.t_pre, "t_0": pe.t_0, "bucket": pe.bucket,
                     "expiry": pe.expiry, "dte": (pe.expiry - pe.t_pre).days, "spot_pre": pe.spot_pre, "K": pe.strikes["K"],
                     "moneyness": pe.strikes["K"] / pe.spot_pre - 1, "call": pe.legs["C_K"].ticker, "put": pe.legs["P_K"].ticker,
                     "call_px": m["C_K"], "put_px": m["P_K"], "implied_move": (m["C_K"] + m["P_K"]) / pe.spot_pre,
                     "spot_gap": pe.synthetic_spot(pe.t_pre, m) / pe.spot_pre - 1,     # this bucket's parity spot vs the near-dated one
                     "atm_volume": pe.legs["C_K"].volume_on(pe.t_pre) + pe.legs["P_K"].volume_on(pe.t_pre)})
    return pd.DataFrame(rows)


events["event_date"] = events["filing_date"]
priced, dropped = price_events(events, label="in-sample")
panel = summarize_priced(priced)
if len(dropped):
    display(dropped["reason"].value_counts().rename("dropped").to_frame())
display(panel.assign(abs_spot_gap=panel.spot_gap.abs()).groupby("bucket")[["dte", "moneyness", "abs_spot_gap", "implied_move", "atm_volume"]]
        .median().round(3).rename(columns=lambda c: f"median {c}"))
with pd.option_context("display.max_colwidth", 40):
    display(panel[panel.bucket == BASELINE_BUCKET][["ticker", "t_pre", "spot_pre", "K", "call", "put", "call_px", "put_px", "implied_move"]].head(6))


  in-sample: 25/36 events, 24s


in-sample: 36 events -> 90 priced (event, bucket) pairs; 14 drops; 41s


,dropped
reason,
1m: ATM pair did not trade on or near the pre-event session,5
2m: ATM pair did not trade on or near the pre-event session,2
2m: no expiry 46-80 days out,2
could not recover spot from the chain (no liquid near-dated pair),2
3-6m: no paired strikes near spot,1
1m: no expiry 21-45 days out,1
3-6m: ATM pair did not trade on or near the pre-event session,1


,median dte,median moneyness,median abs_spot_gap,median implied_move,median atm_volume
bucket,,,,,
1m,30.0,0.000,0.002,0.069,54.0
2m,60.0,0.002,0.004,0.100,237.0
3-6m,115.0,0.001,0.005,0.131,85.0


,ticker,t_pre,spot_pre,K,call,put,call_px,put_px,implied_move
1,CVS,2024-01-04,80.589017,80.0,O:CVS240419C00080000,O:CVS240419P00080000,4.55,3.65,0.101751
4,META,2024-01-31,390.115532,390.0,O:META240517C00390000,O:META240517P00390000,33.80,27.15,0.156236
6,MO,2024-01-31,40.130567,40.0,O:MO240621C00040000,O:MO240621P00040000,1.69,2.20,0.096934
9,DIS,2024-02-06,99.347457,100.0,O:DIS240621C00100000,O:DIS240621P00100000,7.20,6.10,0.133874
12,UBER,2024-02-13,68.947319,70.0,O:UBER240621C00070000,O:UBER240621P00070000,6.00,6.30,0.178397
15,CRM,2024-02-27,299.871386,300.0,O:CRM240621C00300000,O:CRM240621P00300000,25.57,21.45,0.156801


## 6. Profit and loss

Each strategy is a fixed mix of the legs.
Profit and loss is the change of the strategy for each 1 of spot.
If a strategy needs 100 shares, the notebook uses the synthetic stock.


In [9]:
STRATEGIES = ["stock", "long_call", "covered_call", "protective_put", "collar", "cash_secured_put"]
STRATEGY_LABEL = {"stock": "Stock only (synthetic)", "long_call": "1 · Long call", "covered_call": "2 · Covered call",
                  "protective_put": "3 · Protective put", "collar": "4 · Collar", "cash_secured_put": "5 · Cash-secured put"}


def strategy_pnl(m_e: dict, m_x: dict, S_e: float, S_x: float, otm: float) -> dict[str, float]:
    """P&L per $1 of stock at entry, from entry marks `m_e` to exit marks `m_x`.

    The 100 shares behind the covered call, protective put and collar are replaced by the synthetic
    stock (long ATM call, short ATM put), which is what `S_e` and `S_x` are.
    """
    dS = S_x - S_e
    dC_U = m_x[f"C_U{otm}"] - m_e[f"C_U{otm}"]     # the OTM call we sell
    dP_L = m_x[f"P_L{otm}"] - m_e[f"P_L{otm}"]     # the OTM put we buy (or sell, cash-secured)
    dC_K = m_x["C_K"] - m_e["C_K"]                 # the ATM call we buy
    return {
        "stock":            dS / S_e,
        "long_call":        dC_K / S_e,
        "covered_call":     (dS - dC_U) / S_e,
        "protective_put":   (dS + dP_L) / S_e,
        "collar":           (dS + dP_L - dC_U) / S_e,
        "cash_secured_put": (-dP_L) / S_e,
    }


def evaluate(priced: list[PricedEvent], otm_pcts: list[float] = None) -> pd.DataFrame:
    """Long table: one row per (event, bucket, entry, OTM level, horizon) with every strategy's P&L."""
    otm_pcts = otm_pcts or OTM_GRID
    rows = []
    for pe in priced:
        i0 = CAL.get_loc(pe.t_0)
        exits = {0: pe.t_0}
        exits.update({h: CAL[i0 + h] for h in HORIZONS if i0 + h < len(CAL) and CAL[i0 + h] <= pe.expiry_session})
        exits["exp"] = pe.expiry_session
        for entry, e_day in (("pre", pe.t_pre), ("post", pe.t_0)):
            m_e = pe.marks(e_day)
            S_e = pe.synthetic_spot(e_day, m_e)
            if np.isnan(S_e):
                continue
            implied = (m_e["C_K"] + m_e["P_K"]) / S_e
            dte_sessions = sessions_between(e_day, pe.expiry_session)
            for h, x_day in exits.items():
                if x_day > LAST_SESSION:
                    continue
                m_x = pe.marks(x_day)
                S_x = pe.synthetic_spot(x_day, m_x)
                held = sessions_between(e_day, x_day)
                base = {"ticker": pe.ticker, "event_date": pe.event_date, "t_0": pe.t_0, "bucket": pe.bucket, "expiry": pe.expiry,
                        "entry": entry, "entry_date": e_day, "horizon": h, "exit_date": x_day, "sessions_held": held,
                        "dte_sessions": dte_sessions, "S_entry": S_e, "S_exit": S_x, "realized": S_x / S_e - 1,
                        "implied_move": implied,
                        "implied_scaled": implied * np.sqrt(held / dte_sessions) if dte_sessions else np.nan}
                for otm in otm_pcts:
                    rows.append(dict(base, otm=otm, **strategy_pnl(m_e, m_x, S_e, S_x, otm)))
    res = pd.DataFrame(rows)
    res["ratio"] = res["realized"].abs() / res["implied_scaled"]
    return res


results = evaluate(priced)
print(f"{len(results):,} rows: {results.ticker.nunique()} tickers × buckets × 2 entries × {len(OTM_GRID)} OTM levels × horizons")
hz = results[(results.bucket == BASELINE_BUCKET) & (results.entry == ENTRY) & (results.otm == OTM_PCT)]
display(hz.groupby("horizon", sort=False)[STRATEGIES].mean().mul(100).round(2).rename(columns=STRATEGY_LABEL)
        .rename_axis("mean P&L, % of spot at entry").T)


4,692 rows: 24 tickers × buckets × 2 entries × 3 OTM levels × horizons


"mean P&L, % of spot at entry",0,1,2,3,5,10,21,42,63,exp
Stock only (synthetic),0.0,1.67,1.30,1.36,1.90,1.43,2.27,7.35,6.14,5.95
1 · Long call,0.0,0.97,0.66,0.47,0.50,0.46,0.56,2.19,0.41,2.03
2 · Covered call,0.0,0.96,0.83,0.95,1.27,1.15,1.04,5.23,6.33,7.01
3 · Protective put,0.0,1.33,0.75,0.59,1.12,0.76,1.19,5.17,3.10,2.61
4 · Collar,0.0,0.75,0.47,0.38,0.66,0.54,0.54,3.11,3.62,4.40
5 · Cash-secured put,0.0,0.05,0.13,0.17,0.16,0.09,-0.17,0.40,2.01,2.35


## 7. Scoreboard

The scoreboard shows the mean profit and loss for each strategy and each horizon.
The band is the 95 percent bootstrap interval.
A star means that the interval does not contain 0.


In [10]:
def bootstrap_ci(x, n_boot: int = 2000, seed: int = 0) -> tuple[float, float]:
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) < 5:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    means = rng.choice(x, size=(n_boot, len(x)), replace=True).mean(axis=1)
    return tuple(np.percentile(means, [2.5, 97.5]).tolist())


def slice_results(res: pd.DataFrame, bucket: str, entry: str, otm: float) -> pd.DataFrame:
    return res[(res.bucket == bucket) & (res.entry == entry) & (res.otm == otm)]


def scoreboard(res: pd.DataFrame, bucket: str = None, entry: str = None, otm: float = None,
               horizons=None, strategies=STRATEGIES) -> pd.DataFrame:
    """Rows: strategy. Columns: horizon. Cells: mean P&L per $1 of spot, with n and a 95% bootstrap CI."""
    r = slice_results(res, bucket or BASELINE_BUCKET, entry or ENTRY, otm or OTM_PCT)
    horizons = horizons or [h for h in HORIZONS + ["exp"] if h in set(r.horizon)]
    rows = []
    for s in strategies:
        for h in horizons:
            x = r.loc[r.horizon == h, s].dropna()
            lo, hi = bootstrap_ci(x)
            rows.append({"strategy": s, "horizon": h, "n": len(x), "mean": x.mean(), "ci_lo": lo, "ci_hi": hi,
                         "median": x.median(), "hit_rate": (x > 0).mean()})
    return pd.DataFrame(rows)


def difference_board(res_a: pd.DataFrame, res_b: pd.DataFrame, bucket: str = None, entry: str = None, otm: float = None,
                     strategies=STRATEGIES, seed: int = 1) -> pd.DataFrame:
    """Mean P&L of group a minus group b, per strategy and horizon, with a bootstrap CI on the difference."""
    a = slice_results(res_a, bucket or BASELINE_BUCKET, entry or ENTRY, otm or OTM_PCT)
    b = slice_results(res_b, bucket or BASELINE_BUCKET, entry or ENTRY, otm or OTM_PCT)
    rng, rows = np.random.default_rng(seed), []
    for s in strategies:
        for h in [h for h in HORIZONS + ["exp"] if h in set(a.horizon) & set(b.horizon)]:
            xa, xb = a.loc[a.horizon == h, s].dropna().to_numpy(), b.loc[b.horizon == h, s].dropna().to_numpy()
            row = {"strategy": s, "horizon": h, "n_a": len(xa), "n_b": len(xb)}
            if len(xa) >= 5 and len(xb) >= 5:
                d = rng.choice(xa, (2000, len(xa))).mean(1) - rng.choice(xb, (2000, len(xb))).mean(1)
                row.update(mean_a=xa.mean(), mean_b=xb.mean(), difference=xa.mean() - xb.mean(),
                           ci_lo=np.percentile(d, 2.5), ci_hi=np.percentile(d, 97.5))
            rows.append(row)
    return pd.DataFrame(rows)


def fmt_board(board: pd.DataFrame, value: str = "mean", star_if_ci_excludes_zero: bool = True) -> pd.DataFrame:
    """Wide display table: strategies × horizons, values in % of spot, * where the 95% CI excludes zero."""
    def cell(row):
        v = row[value]
        if pd.isna(v):
            return ""
        star = "*" if star_if_ci_excludes_zero and pd.notna(row.get("ci_lo")) and (row["ci_lo"] > 0 or row["ci_hi"] < 0) else ""
        return f"{v * 100:+.2f}%{star}"
    wide = board.assign(cell=board.apply(cell, axis=1)).pivot(index="strategy", columns="horizon", values="cell")
    wide = wide.reindex([s for s in STRATEGIES if s in wide.index]).rename(index=STRATEGY_LABEL)
    wide.columns = [f"h={c}" if c != "exp" else "expiry" for c in wide.columns]
    return wide.rename_axis(f"P&L per $1 spot", axis=0)


INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
SERIES = {"events": "#2a78d6", "placebo": "#eb6834", "oos": "#1baf7a", "in-sample": "#2a78d6"}


def style(ax, title: str, xlabel: str = "", ylabel: str = ""):
    ax.figure.set_facecolor(SURFACE)
    ax.set_facecolor(SURFACE)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(AXIS)
        ax.spines[s].set_linewidth(0.8)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK2, labelsize=9, length=0)
    ax.set_title(title, loc="left", color=INK, fontsize=11, fontweight="bold", pad=10)
    ax.set_xlabel(xlabel, color=INK2, fontsize=9)
    ax.set_ylabel(ylabel, color=INK2, fontsize=9)


def plot_scoreboards(boards: dict[str, pd.DataFrame], title: str, strategies=STRATEGIES):
    """Small multiples: one panel per strategy, one line per group, band = 95% bootstrap CI of the first group."""
    horizons = [h for h in HORIZONS + ["exp"] if any(h in set(b.horizon) for b in boards.values())]
    x = np.arange(len(horizons))
    fig, axes = plt.subplots(2, 3, figsize=(13, 6.6), sharex=True)
    for ax, s in zip(axes.ravel(), strategies):
        for i, (name, b) in enumerate(boards.items()):
            t = b[b.strategy == s].set_index("horizon").reindex(horizons)
            c = SERIES.get(name, INK2)
            if i == 0:
                ax.fill_between(x, t.ci_lo.to_numpy(float) * 100, t.ci_hi.to_numpy(float) * 100, color=c, alpha=0.10, linewidth=0)
            ax.plot(x, t["mean"].to_numpy(float) * 100, color=c, linewidth=2, marker="o", markersize=5,
                    markeredgecolor=SURFACE, markeredgewidth=1.2, linestyle="-" if i == 0 else "--",
                    label=f"{name} (n={int(t.n.max())})")
        ax.axhline(0, color=MUTED, linewidth=0.8)
        ax.set_xticks(x, [str(h) if h != "exp" else "exp" for h in horizons])
        style(ax, STRATEGY_LABEL[s], "sessions after the filing session" if ax in axes[1] else "",
              "mean P&L, % of spot" if ax in axes[:, 0] else "")
    axes[0, 0].legend(frameon=False, labelcolor=INK2, fontsize=8, loc="upper left")
    fig.suptitle(title, x=0.01, ha="left", color=INK, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()


board = scoreboard(results)
print(f"In-sample · {EVENT_TAG} · {BASELINE_BUCKET} options · entry = {ENTRY} · OTM {OTM_PCT:.0%} · "
      f"n = {int(board.n.max())} events. * = 95% bootstrap CI excludes zero.")
display(fmt_board(board))


In-sample · share_repurchase_program · 3-6m options · entry = post · OTM 5% · n = 31 events. * = 95% bootstrap CI excludes zero.


,h=1,h=2,h=3,h=5,h=10,h=21,h=42,h=63,expiry
P&L per $1 spot,,,,,,,,,
Stock only (synthetic),+1.67%,+1.30%,+1.36%,+1.90%*,+1.43%,+2.27%,+7.35%*,+6.14%*,+5.95%*
1 · Long call,+0.97%,+0.66%,+0.47%,+0.50%,+0.46%,+0.56%,+2.19%,+0.41%,+2.03%
2 · Covered call,+0.96%,+0.83%,+0.95%,+1.27%*,+1.15%,+1.04%,+5.23%*,+6.33%*,+7.01%*
3 · Protective put,+1.33%,+0.75%,+0.59%,+1.12%,+0.76%,+1.19%,+5.17%*,+3.10%*,+2.61%
4 · Collar,+0.75%*,+0.47%,+0.38%,+0.66%*,+0.54%,+0.54%,+3.11%*,+3.62%*,+4.40%*
5 · Cash-secured put,+0.05%,+0.13%,+0.17%,+0.16%,+0.09%,-0.17%,+0.40%,+2.01%*,+2.35%*


### Placebo

The placebo uses the same measurement on other sessions for the same companies.
Each placebo session is 30 days or more from an 8-K of that company.
The result is the difference between the events and the placebo.


In [11]:
def sample_placebo(ev: pd.DataFrame, n: int, start: str, end: str, gap_days: int = PLACEBO_GAP_DAYS, seed: int = 7) -> pd.DataFrame:
    """Ordinary sessions for the same tickers, drawn in proportion to how often each ticker has an event,
    at least `gap_days` from any of that ticker's events."""
    rng = np.random.default_rng(seed)
    sessions = CAL[(CAL >= pd.Timestamp(start)) & (CAL <= pd.Timestamp(end))]
    by_ticker = ev.groupby("ticker")["filing_date"].apply(list)
    rows = []
    for t in rng.choice(ev["ticker"].to_numpy(), size=n, replace=True):
        for _ in range(25):
            d = sessions[rng.integers(len(sessions))]
            if all(abs((d - a).days) > gap_days for a in by_ticker.get(t, [])):
                rows.append({"ticker": t, "filing_date": d, "event_date": d, "t_0": d, "t_pre": session_before(d)})
                break
    return pd.DataFrame(rows)


if RUN_PLACEBO:
    placebo_events = sample_placebo(events, N_PLACEBO, STUDY_START, STUDY_END)
    placebo_priced, placebo_dropped = price_events(placebo_events, label="placebo")
    placebo_results = evaluate(placebo_priced)
    placebo_board = scoreboard(placebo_results)
    plot_scoreboards({"events": board, "placebo": placebo_board},
                     f"{EVENT_TAG} vs ordinary days for the same names · {BASELINE_BUCKET} options · entry = {ENTRY} · OTM {OTM_PCT:.0%}")
    diff = difference_board(results, placebo_results)
    print("Events minus ordinary days, mean P&L per $1 spot. * = the 95% interval on the difference excludes zero.")
    display(fmt_board(diff, value="difference"))


  placebo: 25/120 events, 47s


  placebo: 50/120 events, 91s


  placebo: 75/120 events, 131s


  placebo: 100/120 events, 172s


placebo: 120 events -> 312 priced (event, bucket) pairs; 36 drops; 206s


C:\Users\asus\AppData\Local\Temp\ipykernel_12032\2938765321.py:103: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Events minus ordinary days, mean P&L per $1 spot. * = the 95% interval on the difference excludes zero.


,h=1,h=2,h=3,h=5,h=10,h=21,h=42,h=63,expiry
P&L per $1 spot,,,,,,,,,
Stock only (synthetic),+1.29%,+0.48%,+0.29%,+0.49%,-0.20%,-0.53%,+3.07%,+0.45%,-0.10%
1 · Long call,+0.78%,+0.20%,-0.08%,-0.23%,-0.29%,-0.70%,+0.12%,-2.01%,-1.04%
2 · Covered call,+0.68%,+0.30%,+0.25%,+0.36%,+0.09%,-0.71%,+2.49%*,+2.18%,+1.99%
3 · Protective put,+0.98%,+0.05%,-0.19%,+0.04%,-0.32%,-0.76%,+1.66%,-0.53%,-0.54%
4 · Collar,+0.53%,+0.08%,-0.03%,+0.08%,-0.03%,-0.47%,+1.16%,+1.41%,+1.50%
5 · Cash-secured put,-0.06%,-0.08%,-0.15%,-0.22%,-0.40%,-0.75%,-0.38%,+0.43%,+1.02%


### In-sample rank

The function rank_strategies puts the five strategies in order.
The order uses the mean of event minus placebo at horizons 21, 42, and expiry.
This order is a check.
The submission strategy stays the covered call.


In [12]:
def rank_strategies(diff: pd.DataFrame, horizons=(21, 42, "exp")) -> pd.DataFrame:
    """Rank the five strategies by their event-minus-placebo edge, averaged over the given horizons."""
    d = diff[diff.horizon.isin(horizons) & (diff.strategy != "stock")]
    out = (d.groupby("strategy").agg(edge=("difference", "mean"), horizons_ci_excludes_0=("ci_lo", lambda s: int(((s > 0) | (d.loc[s.index, "ci_hi"] < 0)).sum())),
                                     n_events=("n_a", "max"))
             .sort_values("edge", ascending=False))
    out["edge"] = out["edge"].map(lambda v: f"{v * 100:+.2f}%")
    return out.rename(index=STRATEGY_LABEL)


if RUN_PLACEBO:
    ranking = rank_strategies(diff)
    display(ranking)
    top = ranking.index[0]
    print(f"Submission strategy (do not change): '{STRATEGY_LABEL[WINNER]}' "
          f"({BASELINE_BUCKET} options, entry = {ENTRY}, OTM {OTM_PCT:.0%}).")
    print(f"Best event-minus-placebo rank (not used to select): '{top}'. "
          f"Other strategy: '{STRATEGY_LABEL[NEIGHBOR_STRATEGY]}'.")
else:
    print(f"Placebo off; still using frozen '{STRATEGY_LABEL[WINNER]}'.")


,edge,horizons_ci_excludes_0,n_events
strategy,,,
2 · Covered call,+1.25%,1,28
4 · Collar,+0.73%,0,27
3 · Protective put,+0.12%,0,29
5 · Cash-secured put,-0.04%,0,30
1 · Long call,-0.54%,0,31


Frozen submission strategy: '2 · Covered call' (3-6m options, entry = post, OTM 5%).
Largest event-minus-placebo edge (not used to pick): '2 · Covered call'. Predeclared neighbor: '5 · Cash-secured put'.


### Implied move and realized move

The cost of the at-the-money pair on t_pre, divided by spot, is the implied move to expiry.
The synthetic stock gives the realized move.
The ratio is the absolute realized move divided by the implied move.
A ratio near 1 means that the chain was near the true move.


In [13]:
def decay_table(r: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for h in [h for h in HORIZONS + ["exp"] if h in set(r.horizon)]:
        x = r.loc[r.horizon == h, "ratio"].dropna()
        lo, hi = bootstrap_ci(x)
        rows.append({"horizon": h, "n": len(x), "mean_ratio": x.mean(), "ci_lo": lo, "ci_hi": hi,
                     "median_ratio": x.median(), "share_above_1": (x > 1).mean()})
    return pd.DataFrame(rows).set_index("horizon")


def fmt_decay(tbl: pd.DataFrame) -> pd.DataFrame:
    out = tbl.copy()
    for c in ["mean_ratio", "ci_lo", "ci_hi", "median_ratio"]:
        out[c] = out[c].map(lambda v: f"{v:.2f}")
    out["share_above_1"] = out["share_above_1"].map(lambda v: f"{v:.0%}")
    return out


pre = slice_results(results, BASELINE_BUCKET, "pre", OTM_PCT)
pre_exp = pre[pre.horizon == "exp"]
print(f"{BASELINE_BUCKET} ATM pair on the pre-event session: median implied move to expiry {pre_exp.implied_move.median():.1%}, "
      f"median |realized| {pre_exp.realized.abs().median():.1%}; realized beat implied in {(pre_exp.realized.abs() > pre_exp.implied_move).mean():.0%} of events")
tables = {"events": decay_table(pre)}
if RUN_PLACEBO:
    tables["placebo"] = decay_table(slice_results(placebo_results, BASELINE_BUCKET, "pre", OTM_PCT))
display(fmt_decay(pd.concat(tables, names=["group", "horizon"])))


3-6m ATM pair on the pre-event session: median implied move to expiry 13.1%, median |realized| 10.1%; realized beat implied in 38% of events


n mean_ratio ci_lo ci_hi median_ratio share_above_1
group   horizon                                                       
events  1         32       2.05  1.34  2.81         1.30           62%
        2         32       1.57  1.05  2.11         0.94           47%
        3         30       1.40  0.97  1.86         0.96           50%
        5         30       1.25  0.88  1.70         0.87           43%
        10        29       1.27  0.92  1.65         1.15           55%
        21        29       1.08  0.80  1.36         1.01           52%
        42        26       1.09  0.87  1.32         0.98           42%
        63        29       0.79  0.62  0.98         0.72           34%
        exp       31       0.91  0.71  1.14         0.80           39%
placebo 1        110       0.86  0.75  0.99         0.78           38%
        2        110       0.92  0.79  1.06         0.71           33%
        3        111       0.94  0.80  1.08         0.74           38%
        5        111       0.89  0.75  1.03         0.77           35%
        10       108       0.97  0.85  1.10         0.93           45%
        21       106       0.98  0.84  1.11         0.90           41%
        42       106       1.05  0.89  1.21         0.84           43%
        63       102       1.04  0.91  1.19         0.93           44%
        exp       98       1.00  0.87  1.14         0.93           48%

## 8. Out-of-sample

The same pipeline runs on OOS_START to OOS_END.
The in-sample cells do not use these dates.
If a horizon is not complete, the notebook omits that horizon.


In [14]:
oos_events = apply_acceptance_times(build_events(EVENT_TAG, OOS_START, OOS_END, TOP_100))
oos_events["event_date"] = oos_events["filing_date"]
print(f"{len(oos_events)} out-of-sample events across {oos_events.ticker.nunique()} tickers")
display(oos_events[["ticker", "filing_date", "t_pre", "t_0", "days_since_prev_event"]].reset_index(drop=True))
oos_priced, oos_dropped = price_events(oos_events, label="out-of-sample")
oos_results = evaluate(oos_priced)
oos_board = scoreboard(oos_results)
print(f"Out-of-sample · {OOS_START}..{OOS_END} · {BASELINE_BUCKET} options · entry = {ENTRY} · OTM {OTM_PCT:.0%} · n = {int(oos_board.n.max())}")
display(fmt_board(oos_board))

comparison = (board.merge(oos_board, on=["strategy", "horizon"], suffixes=("_in", "_oos"))
                   .assign(same_sign=lambda d: np.sign(d.mean_in) == np.sign(d.mean_oos)))
comp = comparison[comparison.horizon.isin([5, 21, 42, "exp"])]
comp_wide = (comp.assign(cell=lambda d: [f"{a * 100:+.2f}% → {b * 100:+.2f}% {'✓' if s else '✗'}"
                                        for a, b, s in zip(d.mean_in, d.mean_oos, d.same_sign)])
                 .pivot(index="strategy", columns="horizon", values="cell")
                 .reindex(STRATEGIES).rename(index=STRATEGY_LABEL))
comp_wide.columns = [f"h={c}" if c != "exp" else "expiry" for c in comp_wide.columns]
print(f"In-sample mean → out-of-sample mean (per $1 spot); ✓ = same sign. Out-of-sample n = {int(oos_board.n.max())}, "
      f"in-sample n = {int(board.n.max())}.")
display(comp_wide.rename_axis("in → out"))
plot_scoreboards({"in-sample": board, "oos": oos_board},
                 f"{EVENT_TAG}: in-sample ({STUDY_START[:7]}..{STUDY_END[:7]}) vs out-of-sample ({OOS_START[:7]}..{OOS_END[:7]}) · {BASELINE_BUCKET} options")


780 'share_repurchase_program' disclosures across all filers, 2026-01-01..2026-08-31


Acceptance times: 0/14 filings after 16:00 ET; those t_0 moved to the next session.
14 out-of-sample events across 12 tickers


,ticker,filing_date,t_pre,t_0,days_since_prev_event
0,T,2026-01-28,2026-01-27,2026-01-28,NaN
1,NOW,2026-01-28,2026-01-27,2026-01-28,NaN
2,VZ,2026-01-30,2026-01-29,2026-01-30,NaN
3,CRM,2026-03-12,2026-03-11,2026-03-12,NaN
4,INTU,2026-03-16,2026-03-13,2026-03-16,NaN
5,CRM,2026-03-16,2026-03-13,2026-03-16,4.0
6,ADBE,2026-04-21,2026-04-20,2026-04-21,NaN
7,T,2026-04-22,2026-04-21,2026-04-22,84.0
8,NFLX,2026-04-23,2026-04-22,2026-04-23,NaN
9,TMUS,2026-04-23,2026-04-22,2026-04-23,NaN


out-of-sample: 14 events -> 42 priced (event, bucket) pairs; 0 drops; 29s


Out-of-sample · 2026-01-01..2026-08-31 · 3-6m options · entry = post · OTM 5% · n = 13


,h=1,h=2,h=3,h=5,h=10,h=21,h=42,h=63,expiry
P&L per $1 spot,,,,,,,,,
Stock only (synthetic),-0.29%,-0.44%,-0.56%,-0.17%,-0.37%,-1.59%,-6.47%,-13.69%*,-9.01%*
1 · Long call,+0.02%,+0.06%,-0.08%,-0.17%,+0.76%,-1.11%,-2.09%,-6.74%*,-7.26%*
2 · Covered call,-0.13%,-0.22%,-0.49%,-0.11%,-0.94%,-1.90%,-5.83%*,-9.31%*,-3.12%
3 · Protective put,-0.06%,-0.10%,-0.26%,+0.46%,+0.73%,-0.82%,-2.02%,-7.68%*,-8.38%*
4 · Collar,+0.09%,+0.12%,-0.18%,+0.66%,+0.17%,-1.13%,-2.00%,-3.30%*,-2.49%
5 · Cash-secured put,-0.22%,-0.34%,-0.30%,-0.87%,-1.21%,-1.87%,-2.90%,-6.57%*,-2.81%


In-sample mean → out-of-sample mean (per $1 spot); ✓ = same sign. Out-of-sample n = 13, in-sample n = 31.


,h=5,h=21,h=42,expiry
in → out,,,,
Stock only (synthetic),+1.90% → -0.17% ✗,+2.27% → -1.59% ✗,+7.35% → -6.47% ✗,+5.95% → -9.01% ✗
1 · Long call,+0.50% → -0.17% ✗,+0.56% → -1.11% ✗,+2.19% → -2.09% ✗,+2.03% → -7.26% ✗
2 · Covered call,+1.27% → -0.11% ✗,+1.04% → -1.90% ✗,+5.23% → -5.83% ✗,+7.01% → -3.12% ✗
3 · Protective put,+1.12% → +0.46% ✓,+1.19% → -0.82% ✗,+5.17% → -2.02% ✗,+2.61% → -8.38% ✗
4 · Collar,+0.66% → +0.66% ✓,+0.54% → -1.13% ✗,+3.11% → -2.00% ✗,+4.40% → -2.49% ✗
5 · Cash-secured put,+0.16% → -0.87% ✗,-0.17% → -1.87% ✓,+0.40% → -2.90% ✗,+2.35% → -2.81% ✗


C:\Users\asus\AppData\Local\Temp\ipykernel_12032\2938765321.py:103: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. One company

The notebook shows one company as an example.
The company has the most in-sample events among companies that also have an out-of-sample event.
This path is not the scoreboard.


In [15]:
in_counts = events.ticker.value_counts()
cands = [t for t in oos_events.ticker.unique() if t in in_counts.index]
vol = panel[panel.bucket == BASELINE_BUCKET].groupby("ticker")["atm_volume"].median()
if not cands:    # no ticker has events in both windows (rare tag or short window): fall back to the busiest in-sample name
    print(f"No ticker has a '{EVENT_TAG}' event in both the in-sample and out-of-sample windows; "
          "showing the ticker with the most in-sample events instead (no out-of-sample path will be drawn).")
    cands = list(in_counts.index)
assert cands, f"No priced in-sample events for '{EVENT_TAG}'; nothing to illustrate."
EXAMPLE = sorted(cands, key=lambda t: (in_counts[t], vol.get(t, 0)), reverse=True)[0]
print(f"Example ticker: {EXAMPLE} — {in_counts[EXAMPLE]} in-sample share-repurchase 8-Ks and "
      f"{(oos_events.ticker == EXAMPLE).sum()} out-of-sample, median ATM volume {vol.get(EXAMPLE, 0):.0f} contracts on the pre-event session.")

ex_priced = [pe for pe in priced + oos_priced if pe.ticker == EXAMPLE and pe.bucket == BASELINE_BUCKET]
ex_rows = []
for pe in ex_priced:
    sample = "in-sample" if pe.event_date <= pd.Timestamp(STUDY_END) else "out-of-sample"
    m = pe.marks(pe.t_0)
    for name, key in [("ATM call", "C_K"), ("ATM put", "P_K"), (f"call {OTM_PCT:.0%} OTM", f"C_U{OTM_PCT}"), (f"put {OTM_PCT:.0%} OTM", f"P_L{OTM_PCT}")]:
        ex_rows.append({"sample": sample, "filing_date": pe.event_date.date(), "entry": pe.t_0.date(), "expiry": pe.expiry.date(),
                        "spot at entry": round(pe.synthetic_spot(pe.t_0, m), 2), "leg": name, "contract": pe.legs[key].ticker,
                        "strike": pe.legs[key].strike, "close at entry": m[key]})
with pd.option_context("display.max_colwidth", 30, "display.width", 200):
    display(pd.DataFrame(ex_rows).set_index(["sample", "filing_date", "entry", "expiry", "spot at entry", "leg"]))

ex_res = pd.concat([results, oos_results])
ex_res = ex_res[(ex_res.ticker == EXAMPLE) & (ex_res.bucket == BASELINE_BUCKET) & (ex_res.entry == ENTRY) & (ex_res.otm == OTM_PCT)]
ex_tbl = (ex_res[ex_res.horizon.isin([5, 21, 42, "exp"])]
          .assign(sample=lambda d: np.where(d.event_date <= pd.Timestamp(STUDY_END), "in-sample", "out-of-sample"),
                  filing=lambda d: d.event_date.dt.date)
          .set_index(["sample", "filing", "horizon"])[["realized"] + STRATEGIES[1:]]
          .rename(columns=STRATEGY_LABEL).map(lambda v: f"{v * 100:+.1f}%" if pd.notna(v) else ""))
display(ex_tbl)


def pnl_path(pe: PricedEvent, strategy: str, entry: str = ENTRY, otm: float = OTM_PCT) -> pd.Series:
    """The strategy's P&L per $1 spot on every session from entry to expiry."""
    e_day = pe.t_pre if entry == "pre" else pe.t_0
    m_e = pe.marks(e_day)
    S_e = pe.synthetic_spot(e_day, m_e)
    days = CAL[(CAL >= e_day) & (CAL <= min(pe.expiry_session, LAST_SESSION))]
    out = {}
    for d in days:
        m = pe.marks(d)
        out[sessions_between(e_day, d)] = strategy_pnl(m_e, m, S_e, pe.synthetic_spot(d, m), otm)[strategy]
    return pd.Series(out, name=f"{pe.event_date.date()}")


fig, ax = plt.subplots(figsize=(10, 4.8))
for pe in ex_priced:
    sample = "in-sample" if pe.event_date <= pd.Timestamp(STUDY_END) else "oos"
    path = pnl_path(pe, WINNER)
    ax.plot(path.index, path.to_numpy() * 100, color=SERIES[sample], linewidth=2 if sample == "oos" else 1.4,
            alpha=1 if sample == "oos" else 0.85, label=f"{pe.event_date.date()} ({'out-of-sample' if sample == 'oos' else 'in-sample'}, exp {pe.expiry.date()})")
    ax.plot(path.index, pnl_path(pe, "stock").to_numpy() * 100, color=SERIES[sample], linewidth=0.8, linestyle=":", alpha=0.6)
ax.axhline(0, color=MUTED, linewidth=0.8)
style(ax, f"{EXAMPLE}: {STRATEGY_LABEL[WINNER]} after each share-repurchase 8-K · {BASELINE_BUCKET} options · entry = {ENTRY}",
      "sessions after entry", "P&L, % of spot at entry (dotted: the stock alone)")
ax.legend(frameon=False, labelcolor=INK2, fontsize=8)
plt.tight_layout()
plt.show()


Example ticker: CRM — 2 in-sample share-repurchase 8-Ks and 2 out-of-sample, median ATM volume 650 contracts on the pre-event session.


contract  strike  close at entry
sample        filing_date entry      expiry     spot at entry leg                                                      
in-sample     2024-02-28  2024-02-28 2024-06-21 301.13        ATM call     O:CRM240621C00300000   300.0           25.85
                                                              ATM put      O:CRM240621P00300000   300.0           21.00
                                                              call 5% OTM  O:CRM240621C00320000   320.0           17.50
                                                              put 5% OTM   O:CRM240621P00280000   280.0           12.50
              2025-09-03  2025-09-03 2025-12-19 256.63        ATM call     O:CRM251219C00250000   250.0           26.22
                                                              ATM put      O:CRM251219P00250000   250.0           16.68
                                                              call 5% OTM  O:CRM251219C00270000   270.0           16.76
                                                              put 5% OTM   O:CRM251219P00240000   240.0           12.50
out-of-sample 2026-03-12  2026-03-12 2026-07-17 201.62        ATM call     O:CRM260717C00195000   195.0           26.00
                                                              ATM put      O:CRM260717P00195000   195.0           16.68
                                                              call 5% OTM  O:CRM260717C00210000   210.0           16.70
                                                              put 5% OTM   O:CRM260717P00180000   180.0           11.00
              2026-03-16  2026-03-16 2026-07-17 194.44        ATM call     O:CRM260717C00190000   190.0           21.10
                                                              ATM put      O:CRM260717P00190000   190.0           14.12
                                                              call 5% OTM  O:CRM260717C00210000   210.0           13.81
                                                              put 5% OTM   O:CRM260717P00180000   180.0           10.47

realized 1 · Long call 2 · Covered call  \
sample        filing     horizon                                           
in-sample     2024-02-28 5          +1.1%         -0.6%            +2.0%   
                         21         +0.3%         -2.1%            +2.5%   
                         42         -8.6%         -6.8%            -3.5%   
                         exp       -19.0%         -8.6%           -13.2%   
              2025-09-03 5          -5.5%         -4.5%            -2.0%   
                         21         -6.9%         -5.0%            -3.2%   
                         42         +1.7%         -1.3%            +3.2%   
                         exp        +0.9%         -6.7%            +7.5%   
out-of-sample 2026-03-12 5          -3.8%         -3.7%            -1.6%   
                         21        -14.5%         -9.3%            -8.2%   
                         42        -13.1%         -9.5%            -6.5%   
                         exp       -14.9%        -12.9%            -6.6%   
              2026-03-16 5          -0.8%         -0.2%            +0.4%   
                         21         -8.3%         -5.4%            -3.7%   
                         42        -13.8%         -7.7%            -8.2%   
                         exp       -12.2%        -10.8%            -5.1%   

                                 3 · Protective put 4 · Collar  \
sample        filing     horizon                                 
in-sample     2024-02-28 5                    -0.2%      +0.7%   
                         21                   -1.4%      +0.8%   
                         42                   -7.7%      -2.6%   
                         exp                 -11.2%      -5.4%   
              2025-09-03 5                    -4.7%      -1.2%   
                         21                   -5.3%      -1.5%   
                         42                   -0.3%      +1.2%   
                         exp                  -3.9%      +2.6%   
out-of-sample 2026-03-12 5                    -3.5%      -1.2%   
                         21                  -10.9%      -4.5%   
                         42                   -9.2%      -2.7%   
                         exp                 -15.8%      -7.5%   
              2026-03-16 5                    -0.5%      +0.7%   
                         21                   -5.8%      -1.2%   
                         42                   -8.7%      -3.1%   
                         exp                 -12.8%      -5.7%   

                                 5 · Cash-secured put  
sample        filing     horizon                       
in-sample     2024-02-28 5                      +1.3%  
                         21                     +1.7%  
                         42                     -0.9%  
                         exp                    -7.8%  
              2025-09-03 5                      -0.8%  
                         21                     -1.6%  
                         42                     +2.0%  
                         exp                    +4.9%  
out-of-sample 2026-03-12 5                      -0.4%  
                         21                     -3.7%  
                         42                     -3.8%  
                         exp                    +0.9%  
              2026-03-16 5                      -0.3%  
                         21                     -2.5%  
                         42                     -5.1%  
                         exp                    +0.6%

C:\Users\asus\AppData\Local\Temp\ipykernel_12032\248065461.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Sensitivity

A result at one parameter is not enough.
The grid changes OTM_PCT, the expiry bucket, and ENTRY.
The horizon for the grid is 21.


In [16]:
SENS_HORIZON = 21

grid_rows = []
for b in EXPIRY_BUCKETS:
    for e in ("pre", "post"):
        for o in OTM_GRID:
            sb = scoreboard(results, bucket=b, entry=e, otm=o, horizons=[SENS_HORIZON])
            for _, r in sb.iterrows():
                grid_rows.append({"bucket": b, "entry": e, "otm": o, "strategy": r.strategy, "n": r.n, "mean": r["mean"], "ci_lo": r.ci_lo, "ci_hi": r.ci_hi})
grid = pd.DataFrame(grid_rows)
grid["cell"] = [f"{m * 100:+.2f}%{'*' if (lo > 0 or hi < 0) else ''}" if pd.notna(m) else "" for m, lo, hi in zip(grid["mean"], grid.ci_lo, grid.ci_hi)]


def sens_table(sub: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    return (sub[sub.strategy != "stock"].pivot_table(index="strategy", columns=columns, values="cell", aggfunc="first")
              .reindex(STRATEGIES[1:]).rename(index=STRATEGY_LABEL).rename_axis(f"mean P&L at h={SENS_HORIZON}", axis=0))


print(f"1 · Vary the OTM distance of the sold call / bought put, holding the {BASELINE_BUCKET} bucket (in-sample, h={SENS_HORIZON}; * = CI excludes zero):")
display(sens_table(grid[grid.bucket == BASELINE_BUCKET], ["entry", "otm"]))
print(f"2 · Vary the expiry bucket and the entry session, holding OTM = {OTM_PCT:.0%}:")
display(sens_table(grid[grid.otm == OTM_PCT], ["entry", "bucket"]))


1 · Vary the OTM distance of the sold call / bought put, holding the 3-6m bucket (in-sample, h=21; * = CI excludes zero):


entry                   post                     pre                
otm                     0.03    0.05    0.10    0.03    0.05    0.10
mean P&L at h=21                                                    
1 · Long call         +0.56%  +0.56%  +0.56%  +0.69%  +0.69%  +0.69%
2 · Covered call      +1.06%  +1.04%  +1.33%  +1.15%  +1.09%  +1.48%
3 · Protective put    +1.62%  +1.19%  +1.75%  +1.85%  +1.36%  +2.63%
4 · Collar            +0.41%  +0.54%  +0.79%  +0.33%  +0.39%  +1.29%
5 · Cash-secured put  +0.26%  -0.17%  +0.24%  +0.39%  -0.03%  +0.50%

2 · Vary the expiry bucket and the entry session, holding OTM = 5%:


entry                    post                       pre                
bucket                     1m       2m    3-6m       1m      2m    3-6m
mean P&L at h=21                                                       
1 · Long call          -1.08%   +1.07%  +0.56%   -0.24%  +1.22%  +0.69%
2 · Covered call      +2.98%*   +2.02%  +1.04%  +4.00%*  +2.09%  +1.09%
3 · Protective put     -1.12%   +2.14%  +1.19%   +1.02%  +2.35%  +1.36%
4 · Collar             +1.07%  +1.40%*  +0.54%  +2.50%*  +1.35%  +0.39%
5 · Cash-secured put  +0.90%*   +0.58%  -0.17%  +1.36%*  +0.70%  -0.03%

## 11. Trade data

Give the instrument, the entry, the exit, the cost, and the volume.
ENTRY post is the close of t_0 after the acceptance-time rule.
The next cell uses a cost of 5 percent of each option premium on each side.
The next cell also shows the cash-secured put with the same rules.


In [17]:
COST_HAIRCUT = 0.05    # fraction of each option premium paid to get in and out (no quotes in this notebook, so an assumption)

hz = slice_results(results, BASELINE_BUCKET, ENTRY, OTM_PCT)
exp_rows = hz[hz.horizon == 21].copy()
legs_used = {"long_call": ["C_K"], "covered_call": [f"C_U{OTM_PCT}"], "protective_put": [f"P_L{OTM_PCT}"],
             "collar": [f"C_U{OTM_PCT}", f"P_L{OTM_PCT}"], "cash_secured_put": [f"P_L{OTM_PCT}"]}
prem = {(pe.ticker, pe.event_date): pe.marks(pe.t_0) for pe in priced if pe.bucket == BASELINE_BUCKET}


def cost_summary(strategy: str) -> pd.Series:
    vols = {(pe.ticker, pe.event_date): sum(pe.legs[l].volume_on(pe.t_0) for l in legs_used[strategy]) for pe in priced if pe.bucket == BASELINE_BUCKET}
    out = exp_rows.copy()
    out["premium_traded"] = [sum(abs(prem[(t, d)][l]) for l in legs_used[strategy]) / s for t, d, s in zip(out.ticker, out.event_date, out.S_entry)]
    out["cost"] = out["premium_traded"] * COST_HAIRCUT * 2
    out["net"] = out[strategy] - out["cost"]
    out["leg_volume"] = [vols[(t, d)] for t, d in zip(out.ticker, out.event_date)]
    return pd.Series({
        "strategy": STRATEGY_LABEL[strategy],
        "events": f"{len(out)}",
        "gross mean P&L at h=21, per $1 spot": f"{out[strategy].mean() * 100:+.2f}%",
        f"mean round-trip cost ({COST_HAIRCUT:.0%} of premium each way)": f"{out['cost'].mean() * 100:.2f}%",
        "net mean P&L at h=21": f"{out['net'].mean() * 100:+.2f}%",
        "share of events net positive": f"{(out['net'] > 0).mean():.0%}",
        "median contracts traded in the option legs on the filing session": f"{out['leg_volume'].median():,.0f}",
    })


print("Submission strategy:")
display(cost_summary(WINNER).to_frame("value"))
print("Other strategy (same tag, bucket, OTM, and entry):")
display(cost_summary(NEIGHBOR_STRATEGY).to_frame("value"))


Frozen submission strategy:


,value
strategy,2 · Covered call
events,31
"gross mean P&L at h=21, per $1 spot",+1.04%
mean round-trip cost (5% of premium each way),0.44%
net mean P&L at h=21,+0.62%
share of events net positive,52%
median contracts traded in the option legs on the filing session,100


Predeclared neighbor (same tag / bucket / OTM / entry; not selected from the scoreboard):


,value
strategy,5 · Cash-secured put
events,31
"gross mean P&L at h=21, per $1 spot",-0.17%
mean round-trip cost (5% of premium each way),0.35%
net mean P&L at h=21,-0.52%
share of events net positive,52%
median contracts traded in the option legs on the filing session,66


## Timing

filing_date has no time of day.
Each EDGAR header has ACCEPTANCE-DATETIME.
If the 8-K is accepted at 16:00 ET or after, t_0 is the next session.

Section 4 applies this rule before the notebook sets the price.
The function run_study uses the same rule.


In [18]:
# Section 4 sets t_0 before the notebook sets the price.
# This cell only reports the acceptance times.
if "accepted_at" in events.columns:
    ts = pd.to_datetime(events["accepted_at"], errors="coerce")
    after_close = ts.dt.hour.ge(16).fillna(False)
    known = ts.notna()
    print(f"In-sample: {int(known.sum())}/{len(events)} filings had an EDGAR acceptance time; "
          f"{int(after_close.sum())} ({after_close.mean():.0%}) were at or after 16:00 ET and had t_0 moved.")
    if "accepted_at" in oos_events.columns:
        ots = pd.to_datetime(oos_events["accepted_at"], errors="coerce")
        o_after = ots.dt.hour.ge(16).fillna(False)
        print(f"Out-of-sample: {int(ots.notna().sum())}/{len(oos_events)} known; "
              f"{int(o_after.sum())} ({o_after.mean():.0%} of OOS events) after 16:00 ET.")
    show = events.loc[known, ["ticker", "filing_date", "accepted_at", "t_0"]].copy()
    show["after_close"] = after_close[known]
    display(show.head(8).reset_index(drop=True))
else:
    print("Skipped. Set FETCH_ACCEPTANCE_TIMES = True so section 4 can shift post-close filings before pricing.")


In-sample: 0/36 filings had an EDGAR acceptance time; 0 (0%) were at or after 16:00 ET and had t_0 moved.
Out-of-sample: 0/14 known; 0 (0% of OOS events) after 16:00 ET.


,ticker,filing_date,accepted_at,t_0,after_close


## Sealed window

The judge sets RUN_HOLDOUT to True.
The judge sets HOLDOUT_START and HOLDOUT_END in section 2.
Then the judge runs the notebook.

The function run_study uses the tag, the start date, and the end date.
If a horizon is not complete, the notebook omits that horizon.


In [19]:
def run_study(tag: str, start: str, end: str, universe: list[str] = TOP_100, buckets: dict = EXPIRY_BUCKETS,
              max_events: int | None = None, label: str = None) -> dict:
    """The whole pipeline on a fresh window. Returns the events, priced legs, long results and the scoreboard."""
    ev = apply_acceptance_times(build_events(tag, start, end, universe))
    ev["event_date"] = ev["filing_date"]
    if max_events:
        ev = ev.head(max_events).copy()
    pr, drops = price_events(ev, buckets, label=label or f"{tag} {start}..{end}")
    res = evaluate(pr)
    return {"events": ev, "priced": pr, "dropped": drops, "results": res, "board": scoreboard(res)}


if RUN_HOLDOUT:
    holdout = run_study(EVENT_TAG, HOLDOUT_START, HOLDOUT_END)
    print(f"Sealed window {HOLDOUT_START}..{HOLDOUT_END}: n = {int(holdout['board'].n.max())}. "
          "Horizons that have not yet resolved are simply absent.")
    display(fmt_board(holdout["board"]))
    plot_scoreboards({"in-sample": board, "oos": holdout["board"]},
                     f"{EVENT_TAG}: in-sample vs sealed window {HOLDOUT_START}..{HOLDOUT_END} · {BASELINE_BUCKET} options")
else:
    print(f"Sealed window {HOLDOUT_START}..{HOLDOUT_END} not run. Judges: set RUN_HOLDOUT = True.")

# Write scoreboard CSV files. The judge does not need these files.
_tbl = Path("results/tables")
_tbl.mkdir(parents=True, exist_ok=True)
board.to_csv(_tbl / "submission_insample_scoreboard.csv", index=False)
oos_board.to_csv(_tbl / "submission_oos_scoreboard.csv", index=False)
if RUN_PLACEBO:
    placebo_board.to_csv(_tbl / "submission_placebo_scoreboard.csv", index=False)
    diff.to_csv(_tbl / "submission_placebo_diff.csv", index=False)
print(f"Wrote scoreboard CSVs under {_tbl.resolve()}")


Sealed window 2023-06-01..2023-08-31 not run. Judges: set RUN_HOLDOUT = True.
Wrote scoreboard CSVs under D:\dev\CodeOfDuty\Hackathons\gqh-massive-8k-starter-kit\results\tables


## Appendix. Limits

- TOP_100 is a fixed list. The list does not change with time.
- Spot comes from put-call parity. Spot is not a stock bar.
- The synthetic stock has no dividend. The notebook does not include early assignment.
- Each mark is the last trade of the day. The notebook has no bid-ask quote.
- The sample is small.
- The notebook has no earnings calendar.
- filing_date is not the date of the press release.
- If no strike is at the OTM distance, the notebook uses the farthest listed strike.
- Filings and Disclosures start in January 2022.
